In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from ragas.llms import LangchainLLMWrapper
from langchain_deepseek import ChatDeepSeek
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import AnswerAccuracy
from datasets import load_dataset
from copy import deepcopy
import numpy as np
from scipy.stats import ttest_ind
import torch
load_dotenv()

/home/yhuang/ondemand/paper_clean/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [ ]:
df = pd.read_csv('../QA_datasets_classified_qa_eval/output_csv/GoogleNQ_UND_gpt4o_Ragas.csv')
ragas_col = df['ragas_AA_short'].tolist()
pre_ragas = load_dataset("json", 
                        data_files="../QA_datasets_classified_qa_eval/intermediate/BASELINE_GoogleNQ_UND_gpt_with_most_metrics.jsonl",
                        split="all")
assert len(pre_ragas) == len(ragas_col), "Fatal Error: length mismatch"

Generating train split: 458 examples [00:00, 4264.28 examples/s]


In [ ]:
with_ragas = pre_ragas.add_column('ragas_AA_short', ragas_col)
with_ragas.to_json("./intermediate/GoogleNQ_UND_gpt4o_Ragas.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 23.12ba/s]


3208621

## Rewriting with Gemini
GPT-4o rewriting, then GPT-4o QA later

In [ ]:
from helper_functions_qr import modification_in_batch

In [ ]:
client = OpenAI(
    api_key=os.environ.get('GOOGLE_API_KEY'),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)
model = "gemini-2.5-flash"
input_file = "./intermediate/GoogleNQ_UND_gpt4o_Ragas.jsonl"
output_file = "./intermediate/MODIFIED_GoogleNQ_UND_gpt4o_Ragas.jsonl"

In [ ]:
# 清空输出文件（如果存在）
if os.path.exists(output_file):
    os.remove(output_file)
    print(f"Cleared existing output file: {output_file}")

# 处理所有样本（按批次）
question_modification = modification_in_batch(input_file, output_file, 'short_answers', client, model)

Cleared existing output file: ./intermediate/MODIFIED_GoogleNQ_UND_gpt4o_Ragas.jsonl
Total samples to process: 458
Batch size: 3


Processing batches:   0%|          | 0/153 [00:00<?, ?it/s]

Processing batches: 100%|██████████| 153/153 [1:16:26<00:00, 29.98s/it]


All batch processing completed! Total processed: 458 samples
Results saved to: ./intermediate/MODIFIED_GoogleNQ_UND_gpt4o_Ragas.jsonl


In [ ]:
df_view = pd.DataFrame(question_modification)
df_view

,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA
0,where does the modern view of history originat...,"When did the critical, source-based approach t...",[approximately in the early 16th century],[The modern view of history originates from th...,The query asks about the 'modern view of histo...,0.086957,0,0.00
1,when did the first wireless beats come out,When did Beats by Dre release its first wirele...,[October 2012],"[The first wireless Beats, the Beats by Dr. Dr...",The query is ambiguous due to several factors:...,0.142857,0,1.00
2,this inventor co-created the film fred ott’s s...,"Which American inventor, whose pioneering work...",[Edison],[William K.L. Dickson],The query refers to an 'inventor' who co-creat...,0.000000,0,0.25
3,what is the collection of the districts to the...,What are the major geographical or political e...,"[Golan Heights, Jordan]",[Transjordan],"The query refers to 'the Jordan River,' which ...",0.000000,0,0.50
4,factories that assemble parts made in other co...,What are the designated zones or areas that al...,[special economic zones],[Assembly plants],The query lacks specificity regarding critical...,0.000000,0,0.50
...,...,...,...,...,...,...,...,...
453,where do royal families get their money from,What is the historical financial source of the...,[the hereditary revenues of the Crown],[- Inherited wealth and assets \n- Government...,The query is underspecified because 'royal fam...,0.000000,0,0.50
454,where does the show the path take place,Where does the Hulu TV show The Path take place?,[Upstate New York],[The show 'The Path' takes place in Upstate Ne...,The query asks for the location of the show 'T...,0.545455,0,1.00
455,who is the drummer for guns and roses,Who is the current drummer for Guns N' Roses?,[Frank Ferrer],[Frank Ferrer],The query asks for the 'drummer' for Guns N' R...,1.000000,1,1.00
456,what is the meaning of auv in cars,"In the context of cars, what does the abbrevia...",[action utility vehicles],"[Autonomous Underwater Vehicle, Automated Util...",The query asks for the meaning of 'auv' in the...,0.333333,0,0.00


## Modified queries QA using GPT-4o

### Loading modified data

In [ ]:
modified_set = load_dataset("json",
    data_files="./intermediate/MODIFIED_GoogleNQ_UND_gpt4o_Ragas.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)


#modified_set = modified_set.remove_columns(["model_new_answer"])

#modified_set.to_json(
    #"./intermediate/MODIFIED_GoogleNQ_UND_gpt4o_Ragas.jsonl",
    #orient="records",
    #lines=True
#)

Generating train split: 458 examples [00:00, 36981.25 examples/s]


### Implementation

In [ ]:
from helper_functions_qr import (ask_short_answer, run_batch_shortQA_api, batch_QA_with_progress)
client = OpenAI(
    api_key=os.environ.get('OPENAI_API_KEY')
)

In [ ]:
modified_results = batch_QA_with_progress(
    modified_set,
    batch_fn=run_batch_shortQA_api,
    output_key="model_new_answer",
    fill_value=["error"],
    client=client,
    model="gpt-4o-2024-11-20",
    temperature=0.0
)

Running model_new_answer: 100%|██████████| 46/46 [06:44<00:00,  8.79s/it]


In [ ]:
qa_modified = deepcopy(modified_set)
for key in modified_results:
    qa_modified = qa_modified.add_column(key, modified_results[key])

qa_modified.to_json("./intermediate/MODIFIED_GoogleNQ_UND_gpt4o_Ragas.jsonl", orient="records", lines=True)
df_qa_modified = pd.read_json("./intermediate/MODIFIED_GoogleNQ_UND_gpt4o_Ragas.jsonl", lines=True)
#df_qa_modified.to_csv('produced_files/modification_pilot_qa.csv')
df_qa_modified

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 66.85ba/s]


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer
0,where does the modern view of history originat...,"When did the critical, source-based approach t...",[approximately in the early 16th century],[The modern view of history originates from th...,The query asks about the 'modern view of histo...,0.086957,0,0.00,[19th century]
1,when did the first wireless beats come out,When did Beats by Dre release its first wirele...,[October 2012],"[The first wireless Beats, the Beats by Dr. Dr...",The query is ambiguous due to several factors:...,0.142857,0,1.00,[November 2012]
2,this inventor co-created the film fred ott’s s...,"Which American inventor, whose pioneering work...",[Edison],[William K.L. Dickson],The query refers to an 'inventor' who co-creat...,0.000000,0,0.25,[Thomas Edison]
3,what is the collection of the districts to the...,What are the major geographical or political e...,"[Golan Heights, Jordan]",[Transjordan],"The query refers to 'the Jordan River,' which ...",0.000000,0,0.50,[The Hashemite Kingdom of Jordan]
4,factories that assemble parts made in other co...,What are the designated zones or areas that al...,[special economic zones],[Assembly plants],The query lacks specificity regarding critical...,0.000000,0,0.50,"[Free Trade Zones (FTZs), Special Economic Zon..."
...,...,...,...,...,...,...,...,...,...
453,where do royal families get their money from,What is the historical financial source of the...,[the hereditary revenues of the Crown],[- Inherited wealth and assets \n- Government...,The query is underspecified because 'royal fam...,0.000000,0,0.50,"[Crown Estate revenues, Duchy of Lancaster inc..."
454,where does the show the path take place,Where does the Hulu TV show The Path take place?,[Upstate New York],[The show 'The Path' takes place in Upstate Ne...,The query asks for the location of the show 'T...,0.545455,0,1.00,[The Path takes place in Upstate New York.]
455,who is the drummer for guns and roses,Who is the current drummer for Guns N' Roses?,[Frank Ferrer],[Frank Ferrer],The query asks for the 'drummer' for Guns N' R...,1.000000,1,1.00,[Frank Ferrer]
456,what is the meaning of auv in cars,"In the context of cars, what does the abbrevia...",[action utility vehicles],"[Autonomous Underwater Vehicle, Automated Util...",The query asks for the meaning of 'auv' in the...,0.333333,0,0.00,[Asian Utility Vehicle]


## Evaluations

### Squad EM+F1

In [ ]:
# Helper functions updated, RESTART!!
from helper_functions_qr import evaluate_squad_per_sample_multi_ref_pred

modified_set = load_dataset("json",
    data_files="./intermediate/MODIFIED_GoogleNQ_UND_gpt4o_Ragas.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

qa_modified = deepcopy(modified_set)

Generating train split: 458 examples [00:00, 56837.42 examples/s]


In [ ]:
squad_scored_modified, modified_f1_list, modified_em_list = evaluate_squad_per_sample_multi_ref_pred(qa_modified)
squad_scored_modified.to_json("./intermediate/MODIFIED_gpt4o_GoogleNQ_UND_gpt4o_new_squad.jsonl", orient="records", lines=True)

df = pd.read_json("./intermediate/MODIFIED_Gemini_GoogleNQ_UND_gpt4o_new_squad.jsonl", lines=True)
df

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 107.59ba/s]


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer,new_em,new_f1
0,where does the modern view of history originat...,"When did the critical, source-based approach t...",[approximately in the early 16th century],[The modern view of history originates from th...,The query asks about the 'modern view of histo...,0.086957,0,0.00,[19th century],0,0.285714
1,when did the first wireless beats come out,When did Beats by Dre release its first wirele...,[October 2012],"[The first wireless Beats, the Beats by Dr. Dr...",The query is ambiguous due to several factors:...,0.142857,0,1.00,[November 2012],0,0.500000
2,this inventor co-created the film fred ott’s s...,"Which American inventor, whose pioneering work...",[Edison],[William K.L. Dickson],The query refers to an 'inventor' who co-creat...,0.000000,0,0.25,[Thomas Edison],0,0.666667
3,what is the collection of the districts to the...,What are the major geographical or political e...,"[Golan Heights, Jordan]",[Transjordan],"The query refers to 'the Jordan River,' which ...",0.000000,0,0.50,[The Hashemite Kingdom of Jordan],0,0.400000
4,factories that assemble parts made in other co...,What are the designated zones or areas that al...,[special economic zones],[Assembly plants],The query lacks specificity regarding critical...,0.000000,0,0.50,"[Free Trade Zones (FTZs), Special Economic Zon...",0,0.857143
...,...,...,...,...,...,...,...,...,...,...,...
453,where do royal families get their money from,What is the historical financial source of the...,[the hereditary revenues of the Crown],[- Inherited wealth and assets \n- Government...,The query is underspecified because 'royal fam...,0.000000,0,0.50,"[Crown Estate revenues, Duchy of Lancaster inc...",0,0.571429
454,where does the show the path take place,Where does the Hulu TV show The Path take place?,[Upstate New York],[The show 'The Path' takes place in Upstate Ne...,The query asks for the location of the show 'T...,0.545455,0,1.00,[The Path takes place in Upstate New York.],0,0.600000
455,who is the drummer for guns and roses,Who is the current drummer for Guns N' Roses?,[Frank Ferrer],[Frank Ferrer],The query asks for the 'drummer' for Guns N' R...,1.000000,1,1.00,[Frank Ferrer],1,1.000000
456,what is the meaning of auv in cars,"In the context of cars, what does the abbrevia...",[action utility vehicles],"[Autonomous Underwater Vehicle, Automated Util...",The query asks for the meaning of 'auv' in the...,0.333333,0,0.00,[Asian Utility Vehicle],0,0.333333


In [ ]:
modified_mean_em = np.mean(modified_em_list)  # em_scores: EM list per sample
modified_mean_f1 = np.mean(modified_f1_list)  # f1_scores F1 list per sample
print(f"New answers after modification Exact Match (avg): {modified_mean_em * 100:.2f}")
print(f"New answers after modification F1 Score (avg): {modified_mean_f1 * 100:.2f}")

original_em_list = qa_modified['original_em']
original_f1_list = qa_modified['original_f1']

original_mean_em = np.mean(original_em_list)  # em_scores: EM list per sample
original_mean_f1 = np.mean(original_f1_list)  # f1_scores F1 list per sample
print(f"Original answers Exact Match (avg): {original_mean_em * 100:.2f}")
print(f"Original answers F1 Score (avg): {original_mean_f1 * 100:.2f}")

f1_tstat, f1_pval = ttest_ind(modified_f1_list, original_f1_list, equal_var=False)
print(f"F1: t={f1_tstat:.3f}, p={f1_pval:.4f}")

em_tstat, em_pval = ttest_ind(modified_em_list, original_em_list, equal_var=False)
print(f"EM: t={em_tstat:.3f}, p={em_pval:.4f}")

New answers after modification Exact Match (avg): 34.93
New answers after modification F1 Score (avg): 57.33
Original answers Exact Match (avg): 18.78
Original answers F1 Score (avg): 36.98
F1: t=8.112, p=0.0000
EM: t=5.605, p=0.0000


### Ragas AA

In [ ]:
from helper_functions_qr import answer_accuracy_modified
evaluator_llm = LangchainLLMWrapper(ChatDeepSeek(model="deepseek-chat", verbose=True, temperature=0))

In [ ]:
squad_scored_modified = load_dataset("json",
    data_files="./intermediate/MODIFIED_Gemini_GoogleNQ_UND_gpt4o_new_squad.jsonl",
    split="train")
result_with_AA = await answer_accuracy_modified(squad_scored_modified, evaluator_llm)
result_with_AA.to_csv("./output_csv/MODIFIED_Gemini_GoogleNQ_UND_gpt4o_all_new_scores.csv")

Generating train split: 458 examples [00:00, 63443.02 examples/s]
Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 13.63ba/s]


407442

In [ ]:
original_AA = list(result_with_AA["original_AA"])
modified_AA = list(result_with_AA["new_AA"])

original_mean_AA = np.mean(original_AA)
print(f"original AA (avg): {original_mean_AA * 100:.2f}")


modified_mean_AA = np.mean(modified_AA)
print(f"modified AA (avg): {modified_mean_AA * 100:.2f}")

AA_tstat, AA_pval = ttest_ind(modified_AA, original_AA, equal_var=False)
print(f"AA: t={AA_tstat:.3f}, p={AA_pval:.4f}")

original AA (avg): 57.97
modified AA (avg): 73.42
AA: t=5.591, p=0.0000


## Re-Classification

In [2]:
from helper_functions_qr import (
    batch_generate_responses_qwen3,
    get_judgments_from_responses,
    run_experiment,
    prepare_test_prompts)

print("HF_HOME:", os.environ.get("HF_HOME"))
print("HF_DATASETS_CACHE:", os.environ.get("HF_DATASETS_CACHE"))
print("HF_HUB_CACHE:", os.environ.get("HF_HUB_CACHE"))

HF_HOME: /scratch-local/yhuang/huggingface_cache
HF_DATASETS_CACHE: /scratch-local/yhuang/huggingface_cache/datasets
HF_HUB_CACHE: /scratch-local/yhuang/huggingface_cache/hub


### Loading data

In [3]:
reclassify_file = "./output_csv/MODIFIED_Gemini_GoogleNQ_UND_gpt4o_all_new_scores.csv"
df_reclassify_file = pd.read_csv(reclassify_file)

### Loading model

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
Qwen3_4B = "Qwen/Qwen3-4B"
tokenizer = AutoTokenizer.from_pretrained(Qwen3_4B, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(Qwen3_4B)
# 将模型移到可用设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)

Loading checkpoint shards: 100%|██████████| 3/3 [00:12<00:00,  4.27s/it]


cuda


### Prepare prompts

In [5]:
system_prompt = """
You are an expert analyst. Your task is to analyze and determine whether an input user query is "fully specified" or "underspecified".

"""

task_FS_UND = """
Analyze the following input user query:

{"query": "TARGET"}

Please provide your analysis in the following JSON format:

{"query": "TARGET", "reasoning": "[YOUR_DETAILED_REASONING]", "judgment": "[fully specified/underspecified]"}
"""

In [6]:
test_prompts = prepare_test_prompts(df_reclassify_file, task_FS_UND)
print(test_prompts[0])

Start preparing prompts...
# Testing data points: 458
Generation complete: 458 prompts
Average prompt length: 395 bytes (~98 tokens)

Analyze the following input user query:

{"query": "When did the critical, source-based approach to history, often considered the precursor to modern historiography, first emerge?"}

Please provide your analysis in the following JSON format:

{"query": "When did the critical, source-based approach to history, often considered the precursor to modern historiography, first emerge?", "reasoning": "[YOUR_DETAILED_REASONING]", "judgment": "[fully specified/underspecified]"}



In [7]:
test_df = run_experiment(tokenizer, model, test_prompts, system_prompt, df_reclassify_file)
test_df

100%|██████████| 92/92 [53:31<00:00, 34.91s/it]


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer,new_em,new_f1,new_AA,MODIFIED_thinking,MODIFIED_model_response,MODIFIED_model_pred
0,where does the modern view of history originat...,"When did the critical, source-based approach t...",['approximately in the early 16th century'],['The modern view of history originates from t...,The query asks about the 'modern view of histo...,0.086957,0,0.00,['19th century'],0,0.285714,0.00,"<think>\nOkay, let's see. The user is asking w...","{\n ""query"": ""When did the critical, source-b...",fully specified
1,when did the first wireless beats come out,When did Beats by Dre release its first wirele...,['October 2012'],"['The first wireless Beats, the Beats by Dr. D...",The query is ambiguous due to several factors:...,0.142857,0,1.00,['November 2012'],0,0.500000,0.25,"<think>\nOkay, let's see. The user is asking w...","{\n ""query"": ""When did Beats by Dre release i...",fully specified
2,this inventor co-created the film fred ott’s s...,"Which American inventor, whose pioneering work...",['Edison'],['William K.L. Dickson'],The query refers to an 'inventor' who co-creat...,0.000000,0,0.25,['Thomas Edison'],0,0.666667,1.00,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""Which American inventor, whose ...",fully specified
3,what is the collection of the districts to the...,What are the major geographical or political e...,['Golan Heights' 'Jordan'],['Transjordan'],"The query refers to 'the Jordan River,' which ...",0.000000,0,0.50,['The Hashemite Kingdom of Jordan'],0,0.400000,0.75,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""What are the major geographical...",underspecified
4,factories that assemble parts made in other co...,What are the designated zones or areas that al...,['special economic zones'],['Assembly plants'],The query lacks specificity regarding critical...,0.000000,0,0.50,['Free Trade Zones (FTZs)' 'Special Economic Z...,0,0.857143,1.00,"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""What are the designated zones o...",underspecified
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
453,where do royal families get their money from,What is the historical financial source of the...,['the hereditary revenues of the Crown'],['- Inherited wealth and assets \n- Governmen...,The query is underspecified because 'royal fam...,0.000000,0,0.50,['Crown Estate revenues' 'Duchy of Lancaster i...,0,0.571429,0.50,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""What is the historical financia...",underspecified
454,where does the show the path take place,Where does the Hulu TV show The Path take place?,['Upstate New York'],"[""The show 'The Path' takes place in Upstate N...",The query asks for the location of the show 'T...,0.545455,0,1.00,['The Path takes place in Upstate New York.'],0,0.600000,1.00,"<think>\nOkay, let's see. The user is asking w...","{\n ""query"": ""Where does the Hulu TV show T...",fully specified
455,who is the drummer for guns and roses,Who is the current drummer for Guns N' Roses?,['Frank Ferrer'],['Frank Ferrer'],The query asks for the 'drummer' for Guns N' R...,1.000000,1,1.00,['Frank Ferrer'],1,1.000000,1.00,"<think>\nOkay, let's see. The user is asking, ...","{\n ""query"": ""Who is the current drummer for ...",fully specified
456,what is the meaning of auv in cars,"In the context of cars, what does the abbrevia...",['action utility vehicles'],['Autonomous Underwater Vehicle' 'Automated Ut...,The query asks for the meaning of 'auv' in the...,0.333333,0,0.00,['Asian Utility Vehicle'],0,0.333333,0.00,"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""In the context of cars, what do...",underspecified


In [8]:
test_df['MODIFIED_model_pred'].value_counts(normalize=True)


MODIFIED_model_pred
fully specified    0.844978
underspecified     0.155022
Name: proportion, dtype: float64

In [9]:
test_df['MODIFIED_model_pred'].value_counts()

MODIFIED_model_pred
fully specified    387
underspecified      71
Name: count, dtype: int64

In [10]:
test_df.to_csv('./output_csv/GoogleNQ_UND_Gemini_rewritten_reclassified.csv')

## Checking Leakage - Lexical Overlap Analysis

In [1]:
import re
from collections import Counter
import pandas as pd
from helper_functions_qr import (tokenize, get_ngrams, jaccard_similarity,ngram_overlap_f1,compute_metrics)

In [2]:
df_analysis = pd.read_csv('./output_csv/GoogleNQ_UND_Gemini_rewritten_reclassified.csv')

In [3]:
# Original Q-golden A pair
orig_metrics = df_analysis.apply(
    lambda row: compute_metrics(row["original_question"], row["short_answer"]),
    axis=1, result_type="expand"
).add_prefix("orig_")

# Rewritten Q-golden A pair
rewr_metrics = df_analysis.apply(
    lambda row: compute_metrics(row["modified_question"], row["short_answer"]),
    axis=1, result_type="expand"
).add_prefix("rewr_")

results = pd.concat([df_analysis, orig_metrics, rewr_metrics], axis=1)

metrics = ["jaccard", "unigram_f1", "bigram_f1"]
summary = pd.DataFrame({
    "original":  results[[f"orig_{m}" for m in metrics]].mean().values,
    "rewritten": results[[f"rewr_{m}" for m in metrics]].mean().values,
}, index=metrics)
summary["delta"] = summary["rewritten"] - summary["original"]

print(summary.round(4))

            original  rewritten   delta
jaccard       0.0293     0.0477  0.0184
unigram_f1    0.0488     0.0770  0.0282
bigram_f1     0.0049     0.0196  0.0148


In [4]:
from scipy import stats

print("\nWilcoxon signed-rank test (paired, two-sided):")
for m in metrics:
    stat, p = stats.wilcoxon(results[f"orig_{m}"], results[f"rewr_{m}"])
    print(f"  {m:15s}  statistic={stat:.1f}  p={p:.4f}")


Wilcoxon signed-rank test (paired, two-sided):
  jaccard          statistic=3037.0  p=0.0000
  unigram_f1       statistic=3189.0  p=0.0000
  bigram_f1        statistic=180.5  p=0.0000


In [5]:
def cohens_d(a, b):
    diff = a - b
    return diff.mean() / diff.std()

for m in metrics:
    d = cohens_d(results[f"rewr_{m}"], results[f"orig_{m}"])
    print(f"{m:15s}  Cohen's d = {d:.4f}")

jaccard          Cohen's d = 0.2811
unigram_f1       Cohen's d = 0.2925
bigram_f1        Cohen's d = 0.2267
